In [ ]:
"""
constraints.py -- hard constraints and their conversion into QUBO penalties.

Constraint list
  1. No simultaneous charge + discharge:  c*d = 0            (native quadratic penalty)
  2. SOC bounds every slot:               soc_min <= SOC_{v,t} <= soc_max
  3. Required SOC at end of horizon:      SOC_{v,T-1} >= soc_target
  4. Depot charger capacity:              sum_{v in depot}(c+d) <= n_chargers
  5. Transformer limit:                   base + p_c*sum(c) - p_d*sum(d) <= cap_t

Inequalities become equalities with binary-encoded slack bits, then squared:
    a.x <= b   ->   lambda * (a.x - b + s)^2 ,  s in [0, smax]
    a.x >= b   ->   lambda * (a.x - b - s)^2
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import List, Tuple

from variables import Instance, LinExpr, VarIndex, soc_expr, grid_load_expr


class InfeasibleInstance(Exception):
    pass


@dataclass
class LinearConstraint:
    name: str
    expr: LinExpr
    sense: str   # "<=" or ">="
    rhs: float


# --------------------------------------------------------------------------
# Build the constraint list
# --------------------------------------------------------------------------
def _keep(name: str, expr: LinExpr, sense: str, rhs: float):
    """Drop constraints that can never be violated; reject ones that can never hold."""
    lo, hi = expr.bounds()
    if sense == "<=":
        if hi <= rhs:
            return None
        if lo > rhs:
            raise InfeasibleInstance(name)
    else:
        if lo >= rhs:
            return None
        if hi < rhs:
            raise InfeasibleInstance(name)
    return LinearConstraint(name, expr, sense, rhs)


def exclusivity_pairs(inst: Instance, vi: VarIndex) -> List[Tuple[int, int]]:
    pairs = []
    for ev in inst.evs:
        for t in range(inst.T):
            ic, idd = vi.get(f"c_{ev.id}_{t}"), vi.get(f"d_{ev.id}_{t}")
            if ic is not None:
                pairs.append((ic, idd))
    return pairs


def build_constraints(inst: Instance, vi: VarIndex) -> List[LinearConstraint]:
    cons: List[LinearConstraint] = []
    cand = []

    for ev in inst.evs:
        for t in range(inst.T):
            e = soc_expr(inst, vi, ev.id, t)
            cand.append((f"soc_min[{ev.id},{t}]", e, ">=", ev.soc_min))
            cand.append((f"soc_max[{ev.id},{t}]", e, "<=", ev.soc_max))
        cand.append((f"soc_target[{ev.id}]", soc_expr(inst, vi, ev.id, inst.T - 1),
                     ">=", ev.soc_target))

    for dep in inst.depots:
        for t in range(inst.T):
            e = LinExpr()
            for ev in inst.evs:
                if ev.depot == dep.id:
                    e.add(vi.get(f"c_{ev.id}_{t}"), 1)
                    e.add(vi.get(f"d_{ev.id}_{t}"), 1)
            cand.append((f"depot[{dep.id},{t}]", e, "<=", dep.n_chargers))

    for t in range(inst.T):
        cand.append((f"transformer[{t}]", grid_load_expr(inst, vi, t),
                     "<=", inst.transformer_cap[t]))

    for name, e, sense, rhs in cand:
        c = _keep(name, e, sense, rhs)
        if c is not None:
            cons.append(c)
    return cons


# --------------------------------------------------------------------------
# Penalty construction
# --------------------------------------------------------------------------
def slack_weights(smax: int) -> List[int]:
    """Bounded binary encoding: subset sums cover exactly 0..smax."""
    weights, rem, w = [], smax, 1
    while rem > 0:
        take = min(w, rem)
        weights.append(take)
        rem -= take
        w *= 2
    return weights


def add_constraint_penalties(qubo, inst: Instance, vi: VarIndex, lam: float):
    """Add all penalty terms to `qubo`. Appends slack variables to `vi`."""
    # 1. c*d = 0  (exact for binaries)
    for ic, idd in exclusivity_pairs(inst, vi):
        qubo.add_quadratic(ic, idd, lam)

    # 2-5. slack-encoded inequalities
    cons = build_constraints(inst, vi)
    for k, con in enumerate(cons):
        lo, hi = con.expr.bounds()
        smax = (con.rhs - lo) if con.sense == "<=" else (hi - con.rhs)
        if abs(smax - round(smax)) > 1e-9:
            raise ValueError(f"{con.name}: non-integer slack range {smax}; use integer units")
        sign = +1 if con.sense == "<=" else -1
        residual = LinExpr(con.expr.coeffs, con.expr.const - con.rhs)
        for j, w in enumerate(slack_weights(int(round(smax)))):
            s = vi.add(f"s{k}_{j}")
            residual.add(s, sign * w)
        qubo.add_square(residual, lam)
    return cons


# --------------------------------------------------------------------------
# Independent feasibility check (does NOT use the penalty machinery)
# --------------------------------------------------------------------------
def is_feasible_direct(inst: Instance, vi: VarIndex, x) -> bool:
    """Simulate the physical system step by step; x indexed by variable index."""
    def val(name):
        i = vi.get(name)
        return 0 if i is None else x[i]

    soc = {ev.id: ev.soc0 for ev in inst.evs}
    for t in range(inst.T):
        load = inst.base_load[t]
        used = {dep.id: 0 for dep in inst.depots}
        for ev in inst.evs:
            c, d = val(f"c_{ev.id}_{t}"), val(f"d_{ev.id}_{t}")
            if c and d:
                return False
            soc[ev.id] += inst.p_charge * c - inst.p_discharge * d - inst.trip_energy(ev.id, t)
            if not (ev.soc_min <= soc[ev.id] <= ev.soc_max):
                return False
            used[ev.depot] += c + d
            load += inst.p_charge * c - inst.p_discharge * d
        if any(used[dep.id] > dep.n_chargers for dep in inst.depots):
            return False
        if load > inst.transformer_cap[t]:
            return False
    return all(soc[ev.id] >= ev.soc_target for ev in inst.evs)

In [ ]:
"""
objective.py -- Fleet cost + V2G cost/revenue + grid peak penalty.

    f(x) =  sum_{v,t} price_t * p_c * c_{v,t}                    (fleet charging cost)
          - sum_{v,t} feedin_t * p_d * d_{v,t}                   (V2G revenue)
          + sum_{v,t} deg * p_d * d_{v,t}                        (battery degradation)
          + w_peak * sum_t ( base_t + p_c*sum_v c - p_d*sum_v d )^2   (peak shaving)

The last term is a quadratic in the binaries, so it goes straight into Q.
"""
from __future__ import annotations

from variables import Instance, VarIndex, grid_load_expr


def add_objective(qubo, inst: Instance, vi: VarIndex) -> None:
    for ev in inst.evs:
        for t in range(inst.T):
            ic, idd = vi.get(f"c_{ev.id}_{t}"), vi.get(f"d_{ev.id}_{t}")
            if ic is None:
                continue
            qubo.add_linear(ic, inst.price[t] * inst.p_charge)
            qubo.add_linear(idd, (-inst.feedin[t] + inst.deg_cost) * inst.p_discharge)

    if inst.w_peak:
        for t in range(inst.T):
            qubo.add_square(grid_load_expr(inst, vi, t), inst.w_peak)


def evaluate_objective(inst: Instance, vi: VarIndex, x) -> float:
    """Direct (loop-based) evaluation of f(x), independent of the QUBO matrix."""
    def val(name):
        i = vi.get(name)
        return 0 if i is None else x[i]

    total = 0.0
    for t in range(inst.T):
        load = inst.base_load[t]
        for ev in inst.evs:
            c, d = val(f"c_{ev.id}_{t}"), val(f"d_{ev.id}_{t}")
            total += inst.price[t] * inst.p_charge * c
            total += (-inst.feedin[t] + inst.deg_cost) * inst.p_discharge * d
            load += inst.p_charge * c - inst.p_discharge * d
        total += inst.w_peak * load ** 2
    return total

In [ ]:
"""
qubo.py -- assemble  E(x) = x^T Q x + offset  from objective + constraint penalties.

Since x_i^2 = x_i for binaries, linear terms live on the diagonal of Q.
Q is returned upper-triangular (Q[i,j], i<j holds the full pair coefficient).
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Optional, Tuple

import numpy as np

from variables import Instance, VarIndex, LinExpr, build_variables
from objective import add_objective
from constraints import add_constraint_penalties


class QUBO:
    def __init__(self) -> None:
        self.lin: Dict[int, float] = {}
        self.quad: Dict[Tuple[int, int], float] = {}
        self.offset: float = 0.0

    def add_linear(self, i: int, w: float) -> None:
        self.lin[i] = self.lin.get(i, 0.0) + w

    def add_quadratic(self, i: int, j: int, w: float) -> None:
        if i == j:
            self.add_linear(i, w)
            return
        key = (min(i, j), max(i, j))
        self.quad[key] = self.quad.get(key, 0.0) + w

    def add_square(self, expr: LinExpr, weight: float) -> None:
        """Add weight * (const + sum a_i x_i)^2 using x_i^2 = x_i."""
        items = list(expr.coeffs.items())
        c0 = expr.const
        for i, a in items:
            self.add_linear(i, weight * (a * a + 2 * c0 * a))
        for p in range(len(items)):
            for q in range(p + 1, len(items)):
                (i, a), (j, b) = items[p], items[q]
                self.add_quadratic(i, j, weight * 2 * a * b)
        self.offset += weight * c0 * c0

    def abs_sum(self) -> float:
        return sum(abs(v) for v in self.lin.values()) + sum(abs(v) for v in self.quad.values())

    def to_matrix(self, n: int) -> np.ndarray:
        Q = np.zeros((n, n))
        for i, w in self.lin.items():
            Q[i, i] += w
        for (i, j), w in self.quad.items():
            Q[i, j] += w
        return Q


@dataclass
class QUBOModel:
    inst: Instance
    vi: VarIndex
    qubo: QUBO
    Q: np.ndarray
    offset: float
    penalty: float
    constraints: list

    @property
    def n(self) -> int:
        return len(self.vi)

    @property
    def n_decision(self) -> int:
        return self.vi.n_decision

    def energy(self, x) -> float:
        x = np.asarray(x, dtype=float)
        return float(x @ self.Q @ x + self.offset)


def build_qubo(inst: Instance, penalty: Optional[float] = None) -> QUBOModel:
    """
    penalty=None -> automatic lambda = 1 + sum|objective coefficients|.
    All residuals are integers, so any violation costs >= lambda, which exceeds
    the whole objective range: constrained optimum == QUBO optimum.
    (For QAOA you may later try smaller lambda; re-run validate_qubo.py after.)
    """
    vi = build_variables(inst)
    q = QUBO()
    add_objective(q, inst, vi)
    lam = penalty if penalty is not None else 1.0 + q.abs_sum()
    cons = add_constraint_penalties(q, inst, vi, lam)
    return QUBOModel(inst, vi, q, q.to_matrix(len(vi)), q.offset, lam, cons)


# --------------------------------------------------------------------------
# Qiskit hand-off (requires: pip install qiskit qiskit-optimization)
# --------------------------------------------------------------------------
def to_qiskit(model: QUBOModel):
    """Return a qiskit_optimization.QuadraticProgram (all-binary, minimize)."""
    from qiskit_optimization import QuadraticProgram

    qp = QuadraticProgram("ev_grid_qubo")
    for name in model.vi.names:
        qp.binary_var(name)
    names = model.vi.names
    linear = {names[i]: w for i, w in model.qubo.lin.items()}
    quadratic = {(names[i], names[j]): w for (i, j), w in model.qubo.quad.items()}
    qp.minimize(constant=model.offset, linear=linear, quadratic=quadratic)
    return qp


def to_ising(model: QUBOModel):
    """(SparsePauliOp, offset) ready for QAOA / sampling."""
    return to_qiskit(model).to_ising()

In [ ]:
"""
validate_qubo.py -- brute-force proof that the QUBO has the same optimum as the
original constrained problem.

Usage:  python validate_qubo.py [tiny4|small] [--qiskit]

Checks
  A. True optimum: enumerate all 2^n_decision assignments, keep those that pass the
     independent physical feasibility check, evaluate f(x) directly.
  B. QUBO optimum: enumerate all 2^(n_decision + n_slack) assignments of E(x)=x^T Q x + offset.
  C. For every decision assignment, min over slack bits of E must equal f(x) if feasible,
     and be strictly greater than the true optimum if infeasible.
  D. Global QUBO argmin (projected to decision bits) must be a true optimum.
"""
from __future__ import annotations

import sys

import numpy as np

from variables import INSTANCES
from constraints import is_feasible_direct
from objective import evaluate_objective
from qubo import build_qubo, to_qiskit

TOL = 1e-9


def bits(k: int, n: int):
    return [(k >> i) & 1 for i in range(n)]


def all_energies(Q: np.ndarray, offset: float, chunk: int = 1 << 16) -> np.ndarray:
    n = Q.shape[0]
    total = 1 << n
    out = np.empty(total)
    shifts = np.arange(n)
    for start in range(0, total, chunk):
        idx = np.arange(start, min(start + chunk, total))
        X = ((idx[:, None] >> shifts) & 1).astype(float)
        out[start:start + len(idx)] = ((X @ Q) * X).sum(axis=1) + offset
    return out


def main() -> int:
    name = next((a for a in sys.argv[1:] if not a.startswith("--")), "tiny4")
    inst = INSTANCES[name]()
    m = build_qubo(inst)
    nd, n = m.n_decision, m.n
    print(f"instance={name}  decision bits={nd}  slack bits={n - nd}  total={n}  "
          f"penalty lambda={m.penalty:.4f}")
    print(f"constraints kept: {len(m.constraints)}  -> {[c.name for c in m.constraints]}")
    print(f"brute force: 2^{nd} = {1 << nd} decision assignments, 2^{n} = {1 << n} QUBO states\n")

    # ---- A. true constrained optimum --------------------------------------
    true_val = {}
    for k in range(1 << nd):
        x = bits(k, nd)
        if is_feasible_direct(inst, m.vi, x):
            true_val[k] = evaluate_objective(inst, m.vi, x)
    if not true_val:
        print("No feasible assignment exists for this instance.")
        return 1
    opt = min(true_val.values())
    opt_set = {k for k, v in true_val.items() if abs(v - opt) < TOL}
    print(f"[A] feasible assignments: {len(true_val)} / {1 << nd}")
    print(f"    true optimum f* = {opt:.6f} at decision bits "
          f"{[m.vi.names[i] for i in range(nd) if bits(next(iter(opt_set)), nd)[i]]}")

    # ---- B. QUBO brute force ----------------------------------------------
    E = all_energies(m.Q, m.offset)
    best = int(np.argmin(E))
    best_dec = best & ((1 << nd) - 1)
    print(f"[B] QUBO minimum E* = {E[best]:.6f} at decision bits "
          f"{[m.vi.names[i] for i in range(nd) if bits(best_dec, nd)[i]]}")

    # sanity: matrix energy equals dictionary-form energy on random states
    rng = np.random.default_rng(0)
    for k in rng.integers(0, 1 << n, size=min(50, 1 << n)):
        x = bits(int(k), n)
        assert abs(m.energy(x) - E[int(k)]) < 1e-7

    # ---- C. per-decision check --------------------------------------------
    per_dec = E.reshape(1 << (n - nd), 1 << nd).min(axis=0)   # slack bits are the high bits
    bad = 0
    for k in range(1 << nd):
        if k in true_val:
            if abs(per_dec[k] - true_val[k]) > 1e-7:
                bad += 1
        else:
            if per_dec[k] <= opt + 1e-7:
                bad += 1
    print(f"[C] decision assignments whose QUBO value is wrong: {bad}")

    # ---- D. verdict ---------------------------------------------------------
    ok = (abs(E[best] - opt) < 1e-7) and (best_dec in opt_set) and bad == 0
    print(f"[D] QUBO argmin is a true optimum: {best_dec in opt_set};  "
          f"|E* - f*| = {abs(E[best] - opt):.2e}")
    print("\nRESULT:", "PASS - QUBO is equivalent to the constrained problem" if ok else "FAIL")

    if nd <= 4:
        print("\nAll assignments (decision bits -> true f / feasible / min QUBO energy):")
        print("  " + " ".join(m.vi.names[:nd]))
        for k in range(1 << nd):
            f = f"{true_val[k]: .4f}" if k in true_val else "   -   "
            print(f"  {bits(k, nd)}  f={f}  feasible={k in true_val!s:5}  E_min={per_dec[k]: .4f}")

    if "--qiskit" in sys.argv:
        qp = to_qiskit(m)
        print("\nQiskit QuadraticProgram:", qp.get_num_binary_vars(), "binary vars")
        op, off = qp.to_ising()
        print("Ising operator terms:", len(op), " offset:", round(off, 6))

    return 0 if ok else 1


if __name__ == "__main__":
    sys.exit(main())



In [ ]:
"""
variables.py -- problem data, binary variable registry, linear expressions.

Energy is measured in integer "units" (e.g. 1 unit = 10 kWh) so that every
inequality has integer coefficients. That lets us encode slack variables
exactly, which is what makes the QUBO <-> constrained-problem equivalence
provable by brute force.

Decision variables (one pair per EV v and time slot t where the EV is parked):
    c_{v}_{t} = 1  -> EV v charges in slot t   (draws p_charge units from grid)
    d_{v}_{t} = 1  -> EV v discharges (V2G)    (feeds p_discharge units back)
No variable exists while an EV is on a trip (it is fixed to 0 implicitly).
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional


# --------------------------------------------------------------------------
# Problem data
# --------------------------------------------------------------------------
@dataclass(frozen=True)
class EV:
    id: int
    depot: int
    soc0: int        # initial state of charge (units)
    soc_min: int     # battery lower bound (units)
    soc_max: int     # battery upper bound (units)
    soc_target: int  # required SOC at end of horizon (units)


@dataclass(frozen=True)
class Depot:
    id: int
    n_chargers: int  # max simultaneous charge/discharge sessions per slot


@dataclass(frozen=True)
class Trip:
    ev: int
    start: int       # first slot on the road
    end: int         # first slot back at depot (EV away during [start, end))
    energy: int      # units consumed by the trip (booked in slot `start`)


@dataclass
class Instance:
    T: int
    evs: List[EV]
    depots: List[Depot]
    trips: List[Trip]
    price: List[float]            # grid buy price per unit, per slot
    feedin: List[float]           # V2G sell price per unit, per slot
    base_load: List[int]          # non-EV load on the transformer (units/slot)
    transformer_cap: List[int]    # transformer limit (units/slot)
    p_charge: int = 1             # units per charging slot
    p_discharge: int = 1          # units per discharging slot
    deg_cost: float = 0.0         # battery degradation cost per discharged unit
    w_peak: float = 0.0           # weight of the quadratic peak-shaving term

    def on_trip(self, v: int, t: int) -> bool:
        return any(tr.ev == v and tr.start <= t < tr.end for tr in self.trips)

    def trip_energy(self, v: int, t: int) -> int:
        return sum(tr.energy for tr in self.trips if tr.ev == v and tr.start == t)


# --------------------------------------------------------------------------
# Variable registry
# --------------------------------------------------------------------------
class VarIndex:
    """name <-> integer index. Decision variables first, slacks appended later."""

    def __init__(self) -> None:
        self.names: List[str] = []
        self._idx: Dict[str, int] = {}
        self.n_decision: Optional[int] = None

    def add(self, name: str) -> int:
        if name in self._idx:
            raise ValueError(f"duplicate variable {name}")
        self._idx[name] = len(self.names)
        self.names.append(name)
        return self._idx[name]

    def get(self, name: str) -> Optional[int]:
        return self._idx.get(name)

    def __len__(self) -> int:
        return len(self.names)

    def freeze_decision(self) -> None:
        self.n_decision = len(self.names)


def build_variables(inst: Instance) -> VarIndex:
    vi = VarIndex()
    for ev in inst.evs:
        for t in range(inst.T):
            if not inst.on_trip(ev.id, t):
                vi.add(f"c_{ev.id}_{t}")
                vi.add(f"d_{ev.id}_{t}")
    vi.freeze_decision()
    return vi


# --------------------------------------------------------------------------
# Linear expressions over binary variables:  const + sum_i coeff_i * x_i
# --------------------------------------------------------------------------
class LinExpr:
    def __init__(self, coeffs: Optional[Dict[int, float]] = None, const: float = 0):
        self.coeffs: Dict[int, float] = dict(coeffs or {})
        self.const = const

    def add(self, var: Optional[int], coef: float) -> "LinExpr":
        if var is not None:
            self.coeffs[var] = self.coeffs.get(var, 0) + coef
        return self

    def value(self, x) -> float:
        return self.const + sum(a * x[i] for i, a in self.coeffs.items())

    def bounds(self):
        lo = self.const + sum(min(0, a) for a in self.coeffs.values())
        hi = self.const + sum(max(0, a) for a in self.coeffs.values())
        return lo, hi


def soc_expr(inst: Instance, vi: VarIndex, v: int, t: int) -> LinExpr:
    """SOC of EV v at the END of slot t, as a linear function of the binaries.

    SOC_{v,t} = soc0 + sum_{tau<=t} ( p_c*c_{v,tau} - p_d*d_{v,tau} - trip_energy_{v,tau} )
    """
    ev = inst.evs[v]
    e = LinExpr(const=ev.soc0)
    for tau in range(t + 1):
        e.const -= inst.trip_energy(v, tau)
        e.add(vi.get(f"c_{v}_{tau}"), +inst.p_charge)
        e.add(vi.get(f"d_{v}_{tau}"), -inst.p_discharge)
    return e


def grid_load_expr(inst: Instance, vi: VarIndex, t: int) -> LinExpr:
    """Transformer load in slot t = base_load + EV charging - V2G injection."""
    e = LinExpr(const=inst.base_load[t])
    for ev in inst.evs:
        e.add(vi.get(f"c_{ev.id}_{t}"), +inst.p_charge)
        e.add(vi.get(f"d_{ev.id}_{t}"), -inst.p_discharge)
    return e


# --------------------------------------------------------------------------
# Ready-made instances
# --------------------------------------------------------------------------
def make_tiny4() -> Instance:
    """1 EV, 2 slots -> 4 decision bits (16 assignments). V2G in slot 0, recharge in slot 1."""
    return Instance(
        T=2,
        evs=[EV(0, depot=0, soc0=2, soc_min=1, soc_max=4, soc_target=2)],
        depots=[Depot(0, n_chargers=1)],
        trips=[],
        price=[0.30, 0.05],
        feedin=[0.50, 0.10],
        base_load=[3, 3],
        transformer_cap=[5, 5],
        deg_cost=0.02,
        w_peak=0.01,
    )


def make_small() -> Instance:
    """2 EVs, 3 slots, 1 shared charger, 1 trip -> 10 decision bits."""
    return Instance(
        T=3,
        evs=[
            EV(0, depot=0, soc0=3, soc_min=1, soc_max=5, soc_target=2),
            EV(1, depot=0, soc0=1, soc_min=0, soc_max=3, soc_target=3),
        ],
        depots=[Depot(0, n_chargers=1)],
        trips=[Trip(ev=0, start=2, end=3, energy=1)],
        price=[0.30, 0.10, 0.20],
        feedin=[0.45, 0.15, 0.40],
        base_load=[3, 5, 4],
        transformer_cap=[5, 6, 5],
        deg_cost=0.02,
        w_peak=0.02,
    )


INSTANCES = {"tiny4": make_tiny4, "small": make_small}